In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime , timedelta

In [ ]:
API_KEY = '6RvcXqm7yDGUlYuJZFMrFseHTVDAZGzJmmuT9zLl'

start_date = datetime(2015,1,1)
end_date = datetime(2026,5,30)

all_asteroids = []

current = start_date

while current < end_date:
  next_date = current + timedelta(days=7)

  params = {
      'start_date': current.strftime('%Y-%m-%d'),
      'end_date': next_date.strftime('%Y-%m-%d'),
      'api_key' : API_KEY
  }
  response = requests.get(
      "https://api.nasa.gov/neo/rest/v1/feed",
      params=params
  )
  data = response.json()

  for date in data['near_earth_objects']:
      all_asteroids.extend(data['near_earth_objects'][date])


  current = next_date
  print(f'Done : {current.strftime('%Y-%m-%d')}')





Done : 2015-01-08
Done : 2015-01-15
Done : 2015-01-22
Done : 2015-01-29
Done : 2015-02-05
Done : 2015-02-12
Done : 2015-02-19
Done : 2015-02-26
Done : 2015-03-05
Done : 2015-03-12
Done : 2015-03-19
Done : 2015-03-26
Done : 2015-04-02
Done : 2015-04-09
Done : 2015-04-16
Done : 2015-04-23
Done : 2015-04-30
Done : 2015-05-07
Done : 2015-05-14
Done : 2015-05-21
Done : 2015-05-28
Done : 2015-06-04
Done : 2015-06-11
Done : 2015-06-18
Done : 2015-06-25
Done : 2015-07-02
Done : 2015-07-09
Done : 2015-07-16
Done : 2015-07-23
Done : 2015-07-30
Done : 2015-08-06
Done : 2015-08-13
Done : 2015-08-20
Done : 2015-08-27
Done : 2015-09-03
Done : 2015-09-10
Done : 2015-09-17
Done : 2015-09-24
Done : 2015-10-01
Done : 2015-10-08
Done : 2015-10-15
Done : 2015-10-22
Done : 2015-10-29
Done : 2015-11-05
Done : 2015-11-12
Done : 2015-11-19
Done : 2015-11-26
Done : 2015-12-03
Done : 2015-12-10
Done : 2015-12-17
Done : 2015-12-24
Done : 2015-12-31
Done : 2016-01-07
Done : 2016-01-14
Done : 2016-01-21
Done : 201

In [ ]:
df = pd.DataFrame(all_asteroids)
print(df.shape)

df.to_csv("asteroid_final.csv", index=False)

from google.colab import drive
drive.mount('/content/drive')
df.to_csv("/content/drive/MyDrive/asteroid_final.csv", index=False)
print("Drive pe save ho gaya!")

(38573, 11)
Mounted at /content/drive
Drive pe save ho gaya!


In [20]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/asteroid_final.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
df.head()

,links,id,neo_reference_id,name,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,is_sentry_object,sentry_data
0,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,2085990,2085990,85990 (1999 JV6),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.27,{'kilometers': {'estimated_diameter_min': 0.23...,True,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
1,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,2523915,2523915,523915 (1997 VM4),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,18.57,{'kilometers': {'estimated_diameter_min': 0.51...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
2,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3263533,3263533,(2004 XG29),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,25.66,{'kilometers': {'estimated_diameter_min': 0.01...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
3,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3426635,3426635,(2008 RU),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,22.40,{'kilometers': {'estimated_diameter_min': 0.08...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
4,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3601924,3601924,(2012 FO52),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,22.63,{'kilometers': {'estimated_diameter_min': 0.07...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN


In [22]:
df['is_potentially_hazardous_asteroid'].value_counts()

,count
is_potentially_hazardous_asteroid,
False,35170
True,3403


In [23]:
import ast

df['estimated_diameter'] = df['estimated_diameter'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df['close_approach_data'] = df['close_approach_data'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [24]:
df['min_diameter'] = df['estimated_diameter'].apply(
    lambda x : x['kilometers']['estimated_diameter_min']
)

df['max_diameter'] = df['estimated_diameter'].apply(
    lambda x : x['kilometers']['estimated_diameter_max']
)

df['miss_dis_km'] = df['close_approach_data'].apply(
    lambda x : float(x[0]['miss_distance']['kilometers'])
    if len(x) > 0 else None
)

df['velocity_km_s'] = df['close_approach_data'].apply(
    lambda x : float(x[0]['relative_velocity']['kilometers_per_second'])
    if len(x) > 0 else None
)

df['diameter_avg'] = (df['min_diameter'] + df['max_diameter']) / 2

df['threat_score'] = df['velocity_km_s'] / df['miss_dis_km']

df['size_velocity'] = df['diameter_avg'] * df['velocity_km_s']

In [25]:
df.isnull().sum()

,0
links,0
id,0
neo_reference_id,0
name,0
nasa_jpl_url,0
absolute_magnitude_h,0
estimated_diameter,0
is_potentially_hazardous_asteroid,0
close_approach_data,0
is_sentry_object,0


In [26]:
df.drop(columns=['links', 'nasa_jpl_url',
                 'neo_reference_id',
                 'estimated_diameter',
                 'close_approach_data',
                 'sentry_data'], inplace=True)

print(df.shape)
print(df.columns)

(38573, 12)
Index(['id', 'name', 'absolute_magnitude_h',
       'is_potentially_hazardous_asteroid', 'is_sentry_object', 'min_diameter',
       'max_diameter', 'miss_dis_km', 'velocity_km_s', 'diameter_avg',
       'threat_score', 'size_velocity'],
      dtype='object')


In [27]:
df.isnull().sum().sum()

np.int64(0)

In [28]:
df['is_sentry_object'].value_counts()

,count
is_sentry_object,
False,36696
True,1877


In [29]:
df.drop(columns=['is_sentry_object', 'id', 'name'], inplace=True)
print(df.columns)

Index(['absolute_magnitude_h', 'is_potentially_hazardous_asteroid',
       'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s',
       'diameter_avg', 'threat_score', 'size_velocity'],
      dtype='object')


In [36]:
API_KEY = '6RvcXqm7yDGUlYuJZFMrFseHTVDAZGzJmmuT9zLl'
start_date = datetime(2015,1,1)
end_date = datetime(2026,5,30)

url = 'https://api.nasa.gov/DONKI/FLR'
current=start_date


next_date = current + timedelta(days=7)

params = {
      'start_date': current.strftime('%Y-%m-%d'),
      'end_date': next_date.strftime('%Y-%m-%d'),
      'api_key' : API_KEY
  }

response = requests.get(url,params=params)

solar_df = pd.DataFrame(response.json())
print(solar_df.shape)
print(solar_df.head())

(21, 15)
                         flrID      catalog  \
0  2026-05-07T14:20:00-FLR-001  M2M_CATALOG   
1  2026-05-10T13:19:00-FLR-001  M2M_CATALOG   
2  2026-05-13T19:35:00-FLR-001  M2M_CATALOG   
3  2026-05-15T02:52:00-FLR-001  M2M_CATALOG   
4  2026-05-15T13:10:00-FLR-001  M2M_CATALOG   

                                 instruments          beginTime  \
0  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-07T14:20Z   
1  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-10T13:19Z   
2  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-13T19:35Z   
3  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-15T02:52Z   
4  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-15T13:10Z   

            peakTime            endTime classType sourceLocation  \
0  2026-05-07T15:14Z  2026-05-07T15:40Z      M2.6         N18E90   
1  2026-05-10T13:39Z  2026-05-10T14:02Z      M5.7         N21E65   
2  2026-05-13T19:48Z  2026-05-13T19:59Z      C1.4         N18W90   
3  2026-05-15T03:09Z  2026-05-15T

In [37]:
solar_df = solar_df[['flrID', 'beginTime',
                      'classType', 'sourceLocation',
                      'activeRegionNum']]

print(solar_df.shape)
print(solar_df['classType'].value_counts())

(21, 5)
classType
M1.9    2
M1.2    2
M2.6    1
C1.4    1
M5.7    1
C6.7    1
C3.3    1
M1.3    1
M1.4    1
B8.2    1
C9.5    1
M2.3    1
M1.1    1
C1.8    1
C3.1    1
M3.3    1
M9.3    1
M7.7    1
X1.0    1
Name: count, dtype: int64


In [38]:
def flare_risk(classType):
  if classType.startswith('X'):
    return 3
  elif classType.startswith('M'):
    return 2
  elif classType.startswith('C'):
    return 1
  else :
    return 0


solar_df['flare_risk'] = solar_df['classType'].apply(flare_risk)
print(solar_df[['classType', 'flare_risk']])

   classType  flare_risk
0       M2.6           2
1       M5.7           2
2       C1.4           1
3       C3.3           1
4       C6.7           1
5       C9.5           1
6       M1.9           2
7       M1.3           2
8       M1.9           2
9       M1.4           2
10      B8.2           0
11      M2.3           2
12      M1.1           2
13      C1.8           1
14      M1.2           2
15      C3.1           1
16      M1.2           2
17      M3.3           2
18      M9.3           2
19      M7.7           2
20      X1.0           3


In [39]:
avg_flare_risk = solar_df['flare_risk'].mean()
print(f'Average flare risk is {avg_flare_risk}')

Average flare risk is 1.6666666666666667


In [40]:
df['flare_risk'] = avg_flare_risk
print(df.shape)
print(df.head())

(38573, 10)
   absolute_magnitude_h  is_potentially_hazardous_asteroid  min_diameter  \
0                 20.27                               True      0.234723   
1                 18.57                              False      0.513517   
2                 25.66                              False      0.019613   
3                 22.40                              False      0.088015   
4                 22.63                              False      0.079169   

   max_diameter   miss_dis_km  velocity_km_s  diameter_avg  threat_score  \
0      0.524856  1.246329e+07       7.694743      0.379789  6.173925e-07   
1      1.148259  5.815821e+07      16.169622      0.830888  2.780282e-07   
2      0.043857  3.492914e+07      12.530123      0.031735  3.587298e-07   
3      0.196807  2.459521e+07      12.595836      0.142411  5.121255e-07   
4      0.177027  5.554271e+07      15.775064      0.128098  2.840168e-07   

   size_velocity  flare_risk  
0       2.922380    1.666667  
1      13.43

In [41]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'

params = {
    'query' : 'select pl_name , pl_orbsmax , pl_rade , pl_bmasse, st_teff, sy_dist from pscomppars where pl_orbsmax is not null',
    'format' : 'json'
}
response = requests.get(url, params=params)
# print(response.status_code)
# print(response.text[:500])
exo_df = pd.DataFrame(response.json())
print(exo_df.shape)
print(exo_df.head())


(5866, 6)
         pl_name  pl_orbsmax   pl_rade  pl_bmasse  st_teff   sy_dist
0  Kepler-1167 b     0.01750  1.710000      3.570   4971.0   820.905
1  Kepler-1740 b     0.07790  3.323214     11.000   5705.0  1061.770
2  Kepler-1581 b     0.06865  0.800000      0.437   6022.0   493.175
3   Kepler-644 b     0.04641  3.150000     10.100   6747.0  1318.050
4  Kepler-1752 b     0.26980  4.540605     18.700   5446.0   962.888


In [42]:
def habitable_score(row):
  score=0

  if 4000<= row['st_teff'] <= 7000:
    score =+ 1

  if 0.5< row['pl_orbsmax']<= 2.0:
    score+=1

  if 0.5<= row['pl_rade']:
    score += 1

  return score


exo_df['habitable_score'] = exo_df.apply(habitable_score , axis=1)
print(exo_df['habitable_score'].value_counts())

habitable_score
2    4555
1     855
3     452
0       4
Name: count, dtype: int64


In [43]:
avg_habitable = exo_df['habitable_score'].mean()
print(f"Average Habitability Score: {avg_habitable}")

Average Habitability Score: 1.9299352199113535


In [44]:
df['habilability_score'] = avg_habitable

print(df.shape)
print(df.columns.tolist())

(38573, 11)
['absolute_magnitude_h', 'is_potentially_hazardous_asteroid', 'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s', 'diameter_avg', 'threat_score', 'size_velocity', 'flare_risk', 'habilability_score']


In [45]:
X = df[['absolute_magnitude_h', 'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s', 'flare_risk', 'habilability_score', 'diameter_avg', 'threat_score', 'size_velocity']]
y = df['is_potentially_hazardous_asteroid'].astype(int)

In [46]:
print('Befor SMOTE : ' , y.value_counts())

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

Xtrain , Xtest , ytrain , ytest = train_test_split(X, y , random_state=42 , test_size=0.2 , stratify = y)

smote = SMOTE(random_state=42)

Xtrain_smote , ytrain_smote = smote.fit_resample(Xtrain, ytrain)

print("After SMOTE:", pd.Series(ytrain_smote).value_counts())
print("X_train shape:", Xtrain_smote.shape)

Befor SMOTE :  is_potentially_hazardous_asteroid
0    35170
1     3403
Name: count, dtype: int64
After SMOTE: is_potentially_hazardous_asteroid
0    28136
1    28136
Name: count, dtype: int64
X_train shape: (56272, 10)


In [47]:
from xgboost import XGBClassifier
xgbmodel = XGBClassifier(n_estimators=500 , max_depth=5, learning_rate=0.05 , colsample_bytree= 0.8 , random_state=42 , eval_metric='logloss')
xgbmodel.fit(Xtrain_smote , ytrain_smote)

from sklearn.metrics import accuracy_score , f1_score , precision_score, recall_score

ypred = xgbmodel.predict(Xtest)

print('Accuracy score is ' , accuracy_score(ytest, ypred))
print('Precision score is ' , precision_score(ytest, ypred))
print('F1 score is ' , f1_score(ytest, ypred))
print('Recall score is ' , recall_score(ytest, ypred))

Accuracy score is  0.8528839922229423
Precision score is  0.36615566037735847
F1 score is  0.5225073622212874
Recall score is  0.9118942731277533


In [ ]:
# yprobab = xgbmodel.predict_proba(Xtest)[:, 1]

# for threshold in [0.3,0.4,0.5,0.6,0.7,0.8]:
#   ypred_t = (yprobab >= threshold).astype(int)
#   print(f"Threshold {threshold}:")
#   print('Accuracy score is ' , accuracy_score(ytest, ypred_t))
#   print('Precision score is ' , precision_score(ytest, ypred))
#   print('F1 score is ' , f1_score(ytest, ypred))
#   print('Recall score is  ' , recall_score(ytest, ypred))
#   print()

In [ ]:
# print(yprobab[:20])

In [48]:
!pip install plotly

In [49]:
print(df.columns.tolist())
print(df.head(2))
print(df.shape)

['absolute_magnitude_h', 'is_potentially_hazardous_asteroid', 'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s', 'diameter_avg', 'threat_score', 'size_velocity', 'flare_risk', 'habilability_score']
   absolute_magnitude_h  is_potentially_hazardous_asteroid  min_diameter  \
0                 20.27                               True      0.234723   
1                 18.57                              False      0.513517   

   max_diameter   miss_dis_km  velocity_km_s  diameter_avg  threat_score  \
0      0.524856  1.246329e+07       7.694743      0.379789  6.173925e-07   
1      1.148259  5.815821e+07      16.169622      0.830888  2.780282e-07   

   size_velocity  flare_risk  habilability_score  
0       2.922380    1.666667            1.929935  
1      13.435149    1.666667            1.929935  
(38573, 11)


In [50]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df_viz = df.sample(500, random_state=42)
scaler = MinMaxScaler(feature_range=(3, 9))
df_viz['plot_dist'] = scaler.fit_transform(df_viz[['miss_dis_km']])

np.random.seed(42)
theta = np.random.uniform(0, 2*np.pi, len(df_viz))
phi = np.random.uniform(0, np.pi, len(df_viz))
df_viz['ast_x'] = df_viz['plot_dist'] * np.cos(theta) * np.sin(phi)
df_viz['ast_y'] = df_viz['plot_dist'] * np.sin(theta) * np.sin(phi)
df_viz['ast_z'] = df_viz['plot_dist'] * np.cos(phi)

haz = df_viz[df_viz['is_potentially_hazardous_asteroid'] == True]
safe = df_viz[df_viz['is_potentially_hazardous_asteroid'] == False]

u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:60j]

# Earth
ex = np.cos(u)*np.sin(v)
ey = np.sin(u)*np.sin(v)
ez = np.cos(v)

# Shield
sx = 1.6*np.cos(u)*np.sin(v)
sy = 1.6*np.sin(u)*np.sin(v)
sz = 1.6*np.cos(v)

# Safe Zone ring around Earth
sz_theta = np.linspace(0, 2*np.pi, 100)
safe_zone_x = 2.5*np.cos(sz_theta)
safe_zone_y = 2.5*np.sin(sz_theta)
safe_zone_z = np.zeros(100)

# Sun
sun_cx = 35
sunx = 3*np.cos(u)*np.sin(v) + sun_cx
suny = 3*np.sin(u)*np.sin(v)
sunz = 3*np.cos(v)

# Solar panels - lines not squares
sp_theta = np.linspace(0, 2*np.pi, 16)
sp_inner = 4.0
sp_outer = 5.5
sp_x, sp_y, sp_z = [], [], []
for t in sp_theta:
    sp_x += [sp_inner*np.cos(t) + sun_cx, sp_outer*np.cos(t) + sun_cx, None]
    sp_y += [sp_inner*np.sin(t), sp_outer*np.sin(t), None]
    sp_z += [0, 0, None]

# Space Station - safe zone pe (Earth ke paas, hazardous free zone)
ss_x, ss_y, ss_z = 2.5, 0, 0.5

fig = go.Figure()

# Stars background
star_x = np.random.uniform(-20, 30, 300)
star_y = np.random.uniform(-15, 15, 300)
star_z = np.random.uniform(-15, 15, 300)
fig.add_trace(go.Scatter3d(x=star_x, y=star_y, z=star_z,
    mode='markers',
    marker=dict(size=1, color='white', opacity=0.4),
    showlegend=False, hoverinfo='none'))

# Earth
fig.add_trace(go.Surface(x=ex, y=ey, z=ez,
    colorscale=[[0,'#1a5276'],[0.5,'#2e86c1'],[1,'#85c1e9']],
    showscale=False, opacity=1.0, name='🌍 Earth'))

# Shield
fig.add_trace(go.Surface(x=sx, y=sy, z=sz,
    colorscale=[[0,'#00ff88'],[1,'#00ffff']],
    showscale=False, opacity=0.12, name='🛡️ Protection Shield'))

# Safe Zone ring
fig.add_trace(go.Scatter3d(x=safe_zone_x, y=safe_zone_y, z=safe_zone_z,
    mode='lines',
    line=dict(color='#00ff88', width=3),
    name='✅ Safe Zone'))

# Sun
fig.add_trace(go.Surface(x=sunx, y=suny, z=sunz,
    colorscale=[[0,'#e67e22'],[0.5,'#f39c12'],[1,'#f9e79f']],
    showscale=False, opacity=1.0, name='☀️ Sun'))

# Solar Panels - lines
fig.add_trace(go.Scatter3d(x=sp_x, y=sp_y, z=sp_z,
    mode='lines',
    line=dict(color='yellow', width=4),
    name='🔆 Solar Panels'))

# Safe Asteroids
fig.add_trace(go.Scatter3d(
    x=safe['ast_x'], y=safe['ast_y'], z=safe['ast_z'],
    mode='markers',
    marker=dict(size=safe['diameter_avg'].clip(2,8)*2, color='#00ff88', opacity=0.7),
    text=[f"✅ SAFE<br>Dist: {d:.0f}M km<br>Speed: {v:.1f} km/s<br>Size: {s:.2f} km"
          for d,v,s in zip(safe['miss_dis_km']/1e6, safe['velocity_km_s'], safe['diameter_avg'])],
    hoverinfo='text', name='✅ Safe Asteroids'))

# Hazardous Asteroids
fig.add_trace(go.Scatter3d(
    x=haz['ast_x'], y=haz['ast_y'], z=haz['ast_z'],
    mode='markers',
    marker=dict(size=haz['diameter_avg'].clip(2,8)*3, color='#ff4444', opacity=0.9),
    text=[f"⚠️ HAZARDOUS<br>Dist: {d:.0f}M km<br>Speed: {v:.1f} km/s<br>Size: {s:.2f} km"
          for d,v,s in zip(haz['miss_dis_km']/1e6, haz['velocity_km_s'], haz['diameter_avg'])],
    hoverinfo='text', name='☄️ Hazardous Asteroids'))

# Space Station
fig.add_trace(go.Scatter3d(
    x=[ss_x], y=[ss_y], z=[ss_z],
    mode='markers+text',
    marker=dict(size=12, color='cyan', symbol='diamond'),
    text=['🛸 Space Station'],
    textfont=dict(color='cyan', size=12),
    textposition='top center',
    name='🛸 Space Station'))

# Labels
fig.add_trace(go.Scatter3d(
    x=[0, sun_cx],
    z=[2.2, 4.5],
    y=[0, 0],
    mode='text',
    text=['🌍 Earth', '☀️ Sun'],
    textfont=dict(color='white', size=14),
    showlegend=False))

fig.update_layout(
    title=dict(
        text='🛸 AstroShield AI — Space Safety System | Real NASA Data (38,573 Asteroids)',
        font=dict(color='white', size=15)),
    paper_bgcolor='#00000f',
    scene=dict(
        bgcolor='#00000f',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.7)),
        aspectmode='manual',
        aspectratio=dict(x=2, y=1, z=1)
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=700,
    legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,20,0.8)', x=0.01, y=0.99)
)

fig.show()

In [52]:
import pickle

with open('asteroid_model.pkl', 'wb') as f:
    pickle.dump(xgbmodel, f)

print("Model saved!")

Model saved!
